# 00. Data Ingestion & Pipeline Orchestration
**Objective:** Automate the acquisition of base geometries and massive historical parking events, provide guidance for manual portal datasets, and execute the end-to-end ETL spatial matching pipeline to populate the DuckDB database.


### 1. Environment Setup & Dependency Installation
Ensure that all required dependencies (GeoPandas, DuckDB, Pandas, PyYAML, etc.) are installed in your active Jupyter environment.


In [ ]:
import sys
import os
from pathlib import Path

# Path to the .venv in the project root
venv_path = Path("../../.venv")

if not venv_path.exists():
    print("Virtual environment not found. Creating and installing dependencies...")
    !python3 -m venv ../../.venv
    !../../.venv/bin/pip install -r ../../requirements.txt

# Dynamically load venv dependencies into the notebook session
site_packages_dirs = list(venv_path.glob("lib/python*/site-packages"))
if site_packages_dirs:
    site_packages = str(site_packages_dirs[0].resolve())
    if site_packages not in sys.path:
        sys.path.insert(0, site_packages)
        print(f"Dynamically loaded venv dependencies from {site_packages}")


### 2. Download Base GeoJSON Datasets
Automates the retrieval of Bicycle Networks, Parking Bays, and Suburb polygons using paths defined in `config.yaml`.


In [ ]:
import urllib.request
import yaml
from pathlib import Path

# Load project configuration
config_path = Path("../../config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

raw_dir = Path("../../") / config["paths"]["raw_dir"]
raw_dir.mkdir(parents=True, exist_ok=True)

def download_url(url, dest_name):
    dest_path = raw_dir / dest_name
    if dest_path.exists():
        print(f"{dest_name} already exists. Skipping.")
        return
    print(f"Downloading {dest_name}...")
    urllib.request.urlretrieve(url, dest_path)
    print(f"Saved to {dest_path}")

download_url(config["data_sources"]["bicycle_network"], "bicycle_network.geojson")
download_url(config["data_sources"]["parking_bays"], "on_street_parking_bays.geojson")
download_url(config["data_sources"]["suburbs"], "suburbs.geojson")


### 3. Programmatically Download Historical Parking Sensor CSVs
Leverages the direct streaming export capabilities of the Melbourne Open Data API to fetch the 2013 and 2014 massive event logs directly into the extraction paths expected by the pipeline.


In [ ]:
(raw_dir / "parking_2013_extracted").mkdir(exist_ok=True)
(raw_dir / "parking_2014_extracted").mkdir(exist_ok=True)

csv_urls = {
    "parking_2013_extracted/On-street_Car_Parking_Sensor_Data_-_2013.csv": 
        "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-car-parking-sensor-data-2013/exports/csv",
    "parking_2014_extracted/On-street_Car_Parking_Sensor_Data_-_2014.csv": 
        "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-car-parking-sensor-data-2014/exports/csv"
}

for dest, url in csv_urls.items():
    dest_path = raw_dir / dest
    if dest_path.exists():
        print(f"{dest} already exists. Skipping.")
        continue
    print(f"Downloading {dest} from OpenDataSoft API (this is a massive file and will take a few minutes)...")
    urllib.request.urlretrieve(url, dest_path)
    print(f"Saved to {dest_path}")


### 4. Manual Download Instructions for SCATS and VISTA Data
Due to authentication or specific access constraints on the Victorian Government portals, the SCATS Volume and VISTA Survey datasets must be downloaded manually:

1. **SCATS Traffic Volume Data**:
   - Visit the [DataVic Portal](https://discover.data.vic.gov.au/dataset/scats-traffic-signal-volume-data) (or search for SCATS Traffic Signal Volume Data).
   - Export/Download the historical volume CSV files.
   - Place the main volume CSV file exactly at: `data/raw/scats_volume_data.csv`

2. **VISTA Travel Survey Data**:
   - Visit the [Transport Victoria Open Data Portal](https://opendata.transport.vic.gov.au/) and register/login.
   - Search for VISTA (Victorian Integrated Survey of Travel and Activity).
   - Download the recent historical survey data CSV.
   - Place it exactly at: `data/raw/vista_travel_survey.csv`

> **Note:** Because raw SCATS and VISTA schemas vary depending on portal export options, once you place the CSV files, you may need to update the column names in **Notebook 2** and **Notebook 4** to match your exact export headers.


### 5. Execute the End-to-End ETL Pipeline
Once the base GeoJSONs and Parking CSVs are downloaded, and you have placed the SCATS and VISTA files in `data/raw/`, run the cell below to build the complete spatial mappings and the DuckDB analytical database!


In [ ]:
!../../.venv/bin/python ../../run_ingestion.py
